In [ ]:
import os
from os.path import join
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

manuscript_dir = r'C:\Users\Radovan\OneDrive\Radboud\Studentships\Jordy Thielen\Manuscript'
m_ica_dir = join(manuscript_dir, 'data', 'anova', 'ica')
m_noica_dir = join(manuscript_dir, 'data', 'anova', 'noica')

def load_decoding_results(npz_path):
    with np.load(npz_path) as data:
        epoch_bins_loaded = data['subject']
        mean_acc = data['accuracies']
        se_acc = data['ses']
    return epoch_bins_loaded, mean_acc, se_acc

epoch_bins = np.linspace(5, 80, 16, dtype=int)
task = 'covert'


# Full paths
data1_path = os.path.join(m_ica_dir, "covert_lda_alphaCSP_dec_64_ica.npz")
data2_path = os.path.join(m_noica_dir, "covert_lda_alphaCSP_dec_64_noica.npz" )
data3_path = os.path.join(m_ica_dir, "covert_lda_p300_dec_64_ica.npz")
data4_path = os.path.join(m_noica_dir, "covert_lda_p300_dec_64_noica.npz" )
data5_path = os.path.join(m_ica_dir, "covert_lda_rcca_dec_64_ica.npz" )
data6_path = os.path.join(m_noica_dir, "covert_lda_rcca_dec_64_noica.npz" )
# --...---...---...---...---...---...---...---...---...---...---...---...---...-
subj1, mean_acc1, se_acc1 = load_decoding_results(data1_path)
subj2, mean_acc2, se_acc2 = load_decoding_results(data2_path)
subj3, mean_acc3, se_acc3 = load_decoding_results(data3_path)
subj4, mean_acc4, se_acc4 = load_decoding_results(data4_path)
subj5, mean_acc5, se_acc5 = load_decoding_results(data5_path)
subj6, mean_acc6, se_acc6 = load_decoding_results(data6_path)
# --...---...---...---...---...---...---...---...---...---...---...---...---...-
data = {
    'P300 w/ ICA': {
        'mean': mean_acc1,
        'se': se_acc1,
    },
    'P300 wo/ ICA': {
        'mean': mean_acc2,
        'se': se_acc2,
    },
    'Alpha w/ ICA': {
        'mean': mean_acc3,
        'se': se_acc3,
        
    },
    'Alpha wo/ ICA': {
        'mean': mean_acc4,
        'se': se_acc4,
        

},
    'cVEP w/ ICA': {
        'mean': mean_acc5,
        'se': se_acc5,
        
    },
    'cVEP wo/ ICA': {
        'mean': mean_acc6,
        'se': se_acc6,
        

}
}

In [52]:
import pandas as pd
rows = []
n_subjects = subj1.size
for cond_label, acc in data.items():
    paradigm = cond_label.split()[0]   # “P300”, “Alpha”, or “cVEP”
    ica_flag = 'With ICA' if 'w/ ICA' in cond_label else 'No ICA'
    for subj in range(n_subjects):
        rows.append({
            'subject': subj,
            'strategy': paradigm,
            'ICA': ica_flag,
            'accuracy': np.mean(acc['mean'][subj])
        })
df = pd.DataFrame(rows)

In [54]:
from statsmodels.stats.anova import AnovaRM
aov = AnovaRM(df, depvar='accuracy', subject='subject',
                  within=['strategy','ICA']).fit()

In [37]:
print(aov)

                  Anova
             F Value Num DF  Den DF Pr > F
------------------------------------------
paradigm     18.8684 2.0000 56.0000 0.0000
ICA           2.5499 1.0000 28.0000 0.1215
paradigm:ICA  8.1264 2.0000 56.0000 0.0008

